# Identify Retrieval Heads Using Sum Attention Method

### Retrieval Head Identification Using Sum Attention Method

In [2]:
%pip install torch transformers numpy huggingface_hub accelerate bitsandbytes dotenv pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.0 MB/s eta 0:00:00:00:0100:01


In [1]:
!nvidia-smi

Sat Mar 21 01:06:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   28C    P0             68W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os 
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM 
import numpy as np 
import pandas as pd  
import accelerate 
from huggingface_hub import login  
from dotenv import load_dotenv  
import json 
from datetime import datetime 
import hashlib 
import random

#### If using Google Colab: Run The Following Cells (Ignore if using some other environment)

In [ ]:
from google.colab import userdata 

HF_TOKEN = userdata.get('HF_TOKEN') 

In [ ]:
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("HF_TOKEN loaded and Hugging Face login successful.")
else:
    print("HF_TOKEN not found.")

#### If using a high end GPU not from Colab, but from Lambda Labs or Colab GPUs but locally in VSCode

In [6]:
print(f"--- Environment Check ---")
print(f"Remote Working Directory: {os.getcwd()}")

# 2. List all files available to the Python process
print(f"Files in this directory: {os.listdir('.')}")

# 3. Explicitly check for your .env file
env_exists = os.path.exists('.env')
print(f"Is .env visible to the kernel? {env_exists}")

# 4. Try to read a dummy variable if it exists
if env_exists:
    from dotenv import load_dotenv
    load_dotenv()
    print(f"HF_TOKEN found in .env: {'Yes' if os.getenv('HF_TOKEN') else
'No'}")
else:
    print("\n[RESULT] If 'Is .env visible' is False, load_dotenv() will fail.")
    print("You'll need to use Colab Secrets or manually upload your .env file.")

--- Environment Check ---
Remote Working Directory: /content
Files in this directory: ['.config', '.env', 'sample_data']
Is .env visible to the kernel? True
HF_TOKEN found in .env: Yes


In [7]:
# Load HF_TOKEN from a local .env file (works in Lambda Labs / venv / local runs)
load_dotenv(override=False)

# Prefer token from .env or environment; fall back to an existing HF_TOKEN variable (e.g., Colab cell)
HF_TOKEN = os.getenv("HF_TOKEN") or globals().get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("HF_TOKEN loaded and Hugging Face login successful.")
else:
    print("HF_TOKEN not found. Add HF_TOKEN=... to your .env file or set it in the environment.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF_TOKEN loaded and Hugging Face login successful.


#### Configuration & Hyperparameters

In [8]:
import torch

# Model
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

# Data and Needle Stuff

TARGET_SEQ_LEN = 7000

NEEDLE_DEPTH = 0.5
SPLIT = 0.8

# No magic numbers, for reproduction
SPLIT_SEED = 42
NEEDLE_INSERTION_SEED = SPLIT_SEED
TOP_K_HEADS = 20

TASKS = [
    {"id": "registrant_name", "question": "What is the registrant's name?"},
    {"id": "headquarters_city", "question": "What is the registrant's headquarters city?"},
    {"id": "headquarters_state", "question": "What is the registrant's headquarters state?"}, 
    # incorporation state and year should (would it be better to ask for the current state and year instead?)
    {"id": "incorporation_state", "question": "What is the registrant's incorporation state?"},
    {"id": "incorporation_year", "question": "What is the incorporation year?"},  

    {"id": "employees_count_total", "question": "How many total employees does the registrant have?"},
    {"id": "holder_record_amount", "question": "What is the number of holders of record of the registrant's common stock?"},
    {"id": "employees_count_full_time", "question": "How many full-time employees does the registrant have?"},
    {"id": "ceo_lastname", "question": "What is the CEO's last name?"},
] 

TASK_MAP = {t["id"]: t for t in TASKS}

# Paths
DATA_PATH = "data/clean_ground_truth/cleaned_EDGAR_gt_2-22-2026.csv"

# Set up output directories
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
RUN_NAME = f"llama3-8B_instruct_run_{TIMESTAMP}"
BASE_OUTPUT_DIR = f"data/retrieval_heads/01_extractions/{RUN_NAME}"
RAW_OUTPUT_DIR = os.path.join(BASE_OUTPUT_DIR, "raw_tensors")

# Model Loading Options
TORCH_DTYPE = torch.bfloat16
ATTN_IMPL = "eager"

# Formatted by AI to see configs and all that
print("Configuration loaded.")
print(f"Model: {MODEL_ID}")
print(f"Seq length : {TARGET_SEQ_LEN} tokens")
print(f"Needle depth : {NEEDLE_DEPTH * 100:.0f}%")
print(f"ID split: {SPLIT * 100:.0f}% (seed={SPLIT_SEED})")
print(f"Needle insertion seed: {NEEDLE_INSERTION_SEED}")
print(f"Top-K heads: {TOP_K_HEADS}")
print(f"Tasks: {len(TASKS)}")
print(f"dtype: {TORCH_DTYPE}")
print(f"attn_impl: {ATTN_IMPL}")

Configuration loaded.
Model: meta-llama/Meta-Llama-3-8B-Instruct
Seq length : 7000 tokens
Needle depth : 50%
ID split: 80% (seed=42)
Needle insertion seed: 42
Top-K heads: 20
Tasks: 9
dtype: torch.bfloat16
attn_impl: eager



##### Tokenizer & Model Loading

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer loaded: {MODEL_ID}")
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Model max length: {tokenizer.model_max_length:,}")
print(f"Pad token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Tokenizer loaded: meta-llama/Meta-Llama-3-8B-Instruct
Vocab size: 128,000
Model max length: 1,000,000,000,000,000,019,884,624,838,656
Pad token: '<|eot_id|>' (id=128009)


In [10]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=TORCH_DTYPE,
    device_map="auto",
    attn_implementation=ATTN_IMPL,
)
model.eval()

# Formatted by AI to see configs and all that
print(f"Model loaded: {MODEL_ID}")
print(f"dtype: {next(model.parameters()).dtype}")
print(f"Num layers: {model.config.num_hidden_layers}")
print(f"Num heads (Q): {model.config.num_attention_heads}")
print(f"Num KV heads: {model.config.num_key_value_heads}")
print(f"Total heads: {model.config.num_hidden_layers * model.config.num_attention_heads:,}")


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded: meta-llama/Meta-Llama-3-8B-Instruct
dtype: torch.bfloat16
Num layers: 32
Num heads (Q): 32
Num KV heads: 8
Total heads: 1,024


#### Set up Dataset and splitting and all that jazz

In [12]:
# If loading from Vscode  
!pip install scikit-learn

In [14]:
DATA_PATH = "cleaned_EDGAR_gt_2-22-2026.csv"

In [15]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(DATA_PATH)


In [16]:
identification_df, ablation_df = train_test_split(
    df,
    test_size=1 - SPLIT,
    stratify=df["task"],
    random_state=SPLIT_SEED,
)

identification_df = identification_df.reset_index(drop=True)
ablation_df = ablation_df.reset_index(drop=True)

# Asked by AI
print(f"Identification set : {len(identification_df):,} samples ({len(identification_df)/len(df):.0%})")
print(f"Ablation set : {len(ablation_df):,} samples ({len(ablation_df)/len(df):.0%})")


Identification set : 1,061 samples (80%)
Ablation set : 266 samples (20%)


In [17]:
counts = pd.DataFrame({
    "total": df.groupby("task").size(),
    "identification": identification_df.groupby("task").size(),
    "ablation": ablation_df.groupby("task").size(),
})
counts["id_%"] = (counts["identification"] / counts["total"] * 100).round(1)
counts["abl_%"] = (counts["ablation"] / counts["total"] * 100).round(1)

print(counts.to_string())


                           total  identification  ablation  id_%  abl_%
task                                                                   
ceo_lastname                 171             137        34  80.1   19.9
employees_count_full_time     49              39        10  79.6   20.4
employees_count_total        147             118        29  80.3   19.7
headquarters_city            144             115        29  79.9   20.1
headquarters_state           139             111        28  79.9   20.1
holder_record_amount         147             118        29  80.3   19.7
incorporation_state          158             126        32  79.7   20.3
incorporation_year           159             127        32  79.9   20.1
registrant_name              213             170        43  79.8   20.2


#### Building Prompt

In [18]:
def find_needle_span(
    prompt_ids: list[int],
    needle_ids: list[int],
    threshold: float = 0.9,
) -> tuple[int, int]:
    """Locate needle tokens inside the full tokenized prompt via sliding window overlap."""
    span_len = len(needle_ids)
    if span_len == 0:
        return -1, -1

    needle_set = set(needle_ids)

    for i in range(len(prompt_ids) - span_len + 1):
        window = set(prompt_ids[i : i + span_len])
        if len(window & needle_set) / len(needle_set) >= threshold:
            return i, i + span_len

    return -1, -1


#### Deterministic random needle insertion
This extraction pass uses deterministic random insertion per sample (seeded) so insertion positions vary across samples but are reproducible across reruns with the same seed.

In [19]:
def get_deterministic_insertion_index(haystack: str, row: pd.Series, seed: int = NEEDLE_INSERTION_SEED) -> int:
    """Return a deterministic random insertion index per sample."""
    if not haystack:
        return 0

    sample_key = "|".join([
        str(row.get("filename", "")),
        str(row.get("task", "")),
        str(row.get("needle_sentence", "")),
    ])
    digest = hashlib.sha256(f"{seed}|{sample_key}".encode("utf-8")).hexdigest()
    rng = random.Random(int(digest[:16], 16))
    return rng.randint(0, len(haystack))

In [20]:
import re

# It is only <|begin_of_text|> that seems to exist, but I removed all control tokens just to be safe.
CONTROL_TOKENS = ["<|begin_of_text|>", "<|end_of_text|>", "<|eot_id|>", "<|start_header_id|>", "<|end_header_id|>"]
TOKEN_PATTERN = re.compile("|".join(re.escape(t) for t in CONTROL_TOKENS))

def build_prompt(row: pd.Series, task: dict, tokenizer) -> dict: 
    """Construct the full prompt with the needle sentence inserted, and locate the needle span in the tokenized input.""" 
    
    haystack = TOKEN_PATTERN.sub("", row["haystack_text"]).strip()
    needle = row["needle_sentence"]

    insertion_idx = get_deterministic_insertion_index(haystack, row, NEEDLE_INSERTION_SEED)
    context = haystack[:insertion_idx] + " " + needle + " " + haystack[insertion_idx:]

    message = f"<document>{context}</document>\nQuestion: {task['question']}\nAnswer:"
    messages = [{"role": "user", "content": message}]

    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    input_ids = encoded["input_ids"]

    # Locate needle span in the tokenized prompt
    needle_ids = tokenizer.encode(needle, add_special_tokens=False)
    needle_start, needle_end = find_needle_span(input_ids[0].tolist(), needle_ids)

    return {
        "input_ids": input_ids,
        "needle_ids": needle_ids,
        "needle_start": needle_start,
        "needle_end": needle_end,
    }


#### Just some validation of the prompt building logic

In [21]:
import random 
for i in range(5): 
    random_number = random.randint(0, len(identification_df) - 1)
    sample_row = identification_df.iloc[random_number]
    sample_task = TASK_MAP[sample_row["task"]]
    result = build_prompt(sample_row, sample_task, tokenizer)

    print(f"input_ids shape: {result['input_ids'].shape}")
    print(f"needle_start: {result['needle_start']}")
    print(f"needle_end: {result['needle_end']}")
    print(f"needle span len: {result['needle_end'] - result['needle_start']} tokens")
    print(f"total tokens: {result['input_ids'].shape[1]}")
    print()

    decoded = tokenizer.decode(result["input_ids"][0, result["needle_start"]:result["needle_end"]])
    print(f"decoded needle:\n{decoded}")
    print(f"\noriginal needle:\n{sample_row['needle_sentence']}")


input_ids shape: torch.Size([1, 7093])
needle_start: 2216
needle_end: 2285
needle span len: 69 tokens
total tokens: 7093

decoded needle:
 Trading account securities 835 54 The Corporation will furnish to shareholders a copy of any exhibit listed above, but not contained herein, upon written request to Mrs. M. Kitty Jones, Vice President and Secretary, Westamerica Bancorporation, P. O. Box 567, San Rafael, California 94915, and payment to the

original needle:
The Corporation will furnish to shareholders a copy of any exhibit listed above, but not contained herein, upon written request to Mrs. M. Kitty Jones, Vice President and Secretary, Westamerica Bancorporation, P. O. Box 567, San Rafael, California 94915, and payment to the Corporation of $.25 per page.
input_ids shape: torch.Size([1, 7050])
needle_start: 2681
needle_end: 2706
needle span len: 25 tokens
total tokens: 7050

decoded needle:
 As of March 21, 1994, there were approximately 1,900 holders of record of the Common Stock.


#### Run Model & Extract the Attention Weights

In [22]:
def compute_sum_attn(model, prompt_inputs:dict) -> np.ndarray: 
    """ 
    Compute the summed attention scores for each head across all layers, given the prompt inputs.
    """   

    # Just make this a bit more cleaner and easier to read
    input_ids = prompt_inputs["input_ids"].to(model.device)
    needle_start = prompt_inputs["needle_start"]
    needle_end = prompt_inputs["needle_end"]

    with torch.inference_mode():
        prefill = model(
            input_ids=input_ids[:, :-1],
            use_cache=True,
            output_attentions=False,
            return_dict=True,
        )

        decode = model(
            input_ids=input_ids[:, -1:],
            past_key_values=prefill.past_key_values,
            use_cache=False,
            output_attentions=True,
            return_dict=True,
        )

    num_layers = model.config.num_hidden_layers
    num_heads = model.config.num_attention_heads
    scores = np.zeros((num_layers, num_heads), dtype=np.float32)

    for layer_idx, layer_attn in enumerate(decode.attentions):
        attn = layer_attn[0, :, 0, :].float().cpu().numpy() 

        # scores[layer_idx] = attn[:, needle_start:needle_end].mean(axis=1)  
        # we could use mean instead of sum to normalize 
        # for different needle lengths
        scores[layer_idx] = attn[:, needle_start:needle_end].sum(axis=1)

    return scores

Go grab scores for every task

In [23]:
os.makedirs(RAW_OUTPUT_DIR, exist_ok=True)

skipped = 0
total = len(identification_df)

for index, row in identification_df.iterrows():
    if pd.isna(row["haystack_text"]) or pd.isna(row["needle_sentence"]):
        print(f"[{index}/{total}] SKIP - missing text data ({row['filename']})")
        skipped += 1
        continue

    task = TASK_MAP[row["task"]]
    
    filename = f"{row['filename'].replace('.txt', '')}_{task['id']}.npy"
    file_path = os.path.join(RAW_OUTPUT_DIR, filename)
    
    if os.path.exists(file_path):
        print(f"[{index}/{total}] SKIP - already processed {filename}")
        continue

    prompt = build_prompt(row, task, tokenizer)

    if prompt["needle_start"] == -1:
        print(f"[{index}/{total}] SKIP - needle not found ({row['filename']})")
        skipped += 1
        continue

    scores = compute_sum_attn(model, prompt)  
    np.save(file_path, scores)

    print(f"[{index}/{total}] saved {filename}") 
    
    del prompt, scores
    torch.cuda.empty_cache()

print(f"\nDone. {total - skipped}/{total} saved, {skipped} skipped.")

[0/1061] saved 745287_1993_employees_count_total.npy
[1/1061] SKIP - needle not found (1800_1993.txt)
[2/1061] saved 79166_1993_holder_record_amount.npy
[3/1061] saved 105418_1993_incorporation_year.npy
[4/1061] saved 722079_1993_incorporation_year.npy
[5/1061] saved 732714_1993_headquarters_city.npy
[6/1061] saved 277821_1993_headquarters_state.npy
[7/1061] saved 46207_1993_ceo_lastname.npy
[8/1061] saved 64803_1993_ceo_lastname.npy
[9/1061] saved 717605_1993_ceo_lastname.npy
[10/1061] saved 36966_1993_employees_count_full_time.npy
[11/1061] saved 714655_1993_ceo_lastname.npy
[12/1061] saved 859119_1993_incorporation_state.npy
[13/1061] saved 103730_1993_incorporation_year.npy
[14/1061] saved 51296_1993_headquarters_city.npy
[15/1061] saved 29854_1993_ceo_lastname.npy
[16/1061] saved 2024_1993_registrant_name.npy
[17/1061] saved 277577_1993_ceo_lastname.npy
[18/1061] saved 72971_1993_employees_count_total.npy
[19/1061] saved 793548_1993_holder_record_amount.npy
[20/1061] saved 764037_

In [24]:
import json
meta = {
    "model_id": MODEL_ID,
    "target_seq_len": TARGET_SEQ_LEN,
    "needle_depth": NEEDLE_DEPTH,
    "split": SPLIT,
    "split_seed": SPLIT_SEED,
    "timestamp": TIMESTAMP
}
with open(os.path.join(BASE_OUTPUT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Saved meta.json")


Saved meta.json


In [25]:
!zip -r "{BASE_OUTPUT_DIR}.zip" "{BASE_OUTPUT_DIR}"

  adding: data/retrieval_heads/01_extractions/llama3-8B_instruct_run_2026-03-21_01-10-12/ (stored 0%)
  adding: data/retrieval_heads/01_extractions/llama3-8B_instruct_run_2026-03-21_01-10-12/meta.json (deflated 24%)
  adding: data/retrieval_heads/01_extractions/llama3-8B_instruct_run_2026-03-21_01-10-12/raw_tensors/ (stored 0%)
  adding: data/retrieval_heads/01_extractions/llama3-8B_instruct_run_2026-03-21_01-10-12/raw_tensors/83047_1993_headquarters_city.npy (deflated 8%)
  adding: data/retrieval_heads/01_extractions/llama3-8B_instruct_run_2026-03-21_01-10-12/raw_tensors/66740_1993_incorporation_year.npy (deflated 9%)
  adding: data/retrieval_heads/01_extractions/llama3-8B_instruct_run_2026-03-21_01-10-12/raw_tensors/201461_1993_headquarters_city.npy (deflated 9%)
  adding: data/retrieval_heads/01_extractions/llama3-8B_instruct_run_2026-03-21_01-10-12/raw_tensors/9892_1993_headquarters_city.npy (deflated 13%)
  adding: data/retrieval_heads/01_extractions/llama3-8B_instruct_run_2026-03